# Portfolio


## Settings

- **Purpose:** Define the single meta-filtered account.
- **Settings:** Initial assets=100000 USD; K=5 per direction; fee=1 bp and slippage=1 bp per side.
- **Data:** Shared immutable defaults and frozen price calibration.
- **Decision:** The final statistics notebook performs the full simulation once; this notebook writes no ledger.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from src.preprocessing.market_data import ResearchPaths
from src.backtesting.portfolio import PortfolioSettings
PROJECT_ROOT = Path.cwd().resolve().parents[1]
paths = ResearchPaths(PROJECT_ROOT)
settings = PortfolioSettings()
from src.backtesting.portfolio import load_prediction_events
from src.backtesting.portfolio import candidate_snapshot
from src.preprocessing.market_data import sessions
calibration = pd.read_parquet(paths.artifacts / "price_calibration.parquet")
events = load_prediction_events(paths)
display(pd.Series(vars(settings), name="value").to_frame())


## Candidates and Weights

- **Purpose:** Inspect a development-only snapshot without simulating an account.
- **Settings:** Latest active event ranks each security; all its active event signals determine its weight.
- **Data:** At the first development timestamp, only newly arriving events are active.
- **Decision:** Top and bottom floor(N*20%) candidates require at least K members; unavailable slots remain cash.


In [ ]:
development = events.loc[events.partition.eq("development")]
example_time = development.event_start.min()
example_active = development.loc[development.event_start.eq(example_time)].copy()
prices = {}
for symbol in example_active.symbol.unique():
    bars = pd.read_parquet(paths.feature(symbol, "dollar_bars")).set_index("end")
    prices[symbol] = float(bars.loc[example_time, "close"])
example_active["entry_price"] = example_active.symbol.map(prices)
# This snapshot is descriptive only. Price calibration is in holdout-start units,
# so inspect probability targets without combining incompatible price units.
from src.backtesting.bet_sizing import average_symbol_targets
display(average_symbol_targets(example_active, settings.k, settings.step_size).to_frame())


## Queues and Replacement

- **Purpose:** Document the causal account transition rules.
- **Settings:** Positive score; 390 regular-session-minute half-life; incumbent wins ties.
- **Data:** Runtime active events, executable quotes, and separate long/short queues.
- **Decision:** Observed event expiry changes only that security’s average. Replacement requires executable quotes for both legs after the decision; no speculative incumbent sale.


In [ ]:
rules = pd.Series({
    "sizing": "transform each event; average within symbol including zero pass signals; discretize",
    "new_or_increased_position": "tail eligibility, aligned latest direction, take, positive score, limit price",
    "reduction": "no entry limit; execute after the decision",
    "replacement": "strictly higher score; both legs executable in the same quote batch",
    "final_liquidation": "request exits one regular-session minute before final close",
    "corporate_actions": "outside this simplified fixed-symbol evaluation",
    "saved_results": "backtest_statistics.parquet only",
}, name="policy")
display(rules.to_frame())
